# Fase 5 - Patrones de Demanda por Zona: Reposicionamiento de Flota

**Pregunta de negocio:** ¿Podemos anticipar zonas de alta demanda para reposicionar la flota?

## Objetivos de este notebook

1. Analizar la demanda de taxis por zona geográfica y hora del día
2. Crear matrices de demanda (zona x hora, zona x día de la semana)
3. Construir features predictivos con variables rezagadas y medias móviles
4. Entrenar un Random Forest para predecir demanda por zona
5. Generar recomendaciones de reposicionamiento de flota
6. Estimar el valor de negocio de una distribución óptima de taxis

## ¿Por qué importa el reposicionamiento de flota?

En cualquier momento, hay zonas con **exceso de taxis** (oferta > demanda) y zonas con
**escasez** (demanda > oferta). Si podemos predecir dónde estará la demanda en la
próxima hora, podemos mover taxis proactivamente de zonas de baja a alta demanda.

Beneficios:
- **Pasajeros:** Menor tiempo de espera
- **Conductores:** Más viajes por turno, mayor ingreso
- **Empresa:** Mayor utilización de la flota, más ingresos totales

## 1. Configuración e importaciones

In [ ]:
# Conexión a BigQuery
import sys
sys.path.insert(0, '../../../src')
from bigquery.bq_helper import BigQueryHelper
bq = BigQueryHelper()

In [ ]:
# Librerías de análisis y visualización
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Machine Learning
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import LabelEncoder

# Para guardar modelos
import pickle
import os

# Configuración de estilo
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

## 2. Consulta de demanda por zona y hora

Definimos zonas geograficas basadas en el `pickup_location_id` (TLC Zone ID, 1-263):
- **Lower Manhattan:** zonas del sur de Manhattan (Financial District, SoHo, Tribeca)
- **Midtown:** zonas centrales de Manhattan (Times Square, Empire State, Grand Central)
- **Upper East Side:** zonas del este alto de Manhattan
- **Upper West Side:** zonas del oeste alto de Manhattan
- **Uptown:** zonas del norte de Manhattan (Harlem, Washington Heights)

Agrupamos por zona, fecha y hora para obtener la demanda granular.

In [ ]:
query = """
SELECT
    DATE(pickup_datetime) AS fecha,
    EXTRACT(HOUR FROM pickup_datetime) AS hora,
    EXTRACT(DAYOFWEEK FROM pickup_datetime) AS day_of_week,
    EXTRACT(MONTH FROM pickup_datetime) AS month,
    CASE
        WHEN pickup_location_id IN ('12','13','87','88','209','231','261') THEN 'Lower Manhattan'
        WHEN pickup_location_id IN ('48','50','68','100','125','127','128','137','140','141','142','143','144','151','158','161','162','163','164','170','186','230','234','236','237','239','243','244','246','249') THEN 'Midtown'
        WHEN pickup_location_id IN ('43','74','75','79','107','113','114','116','148','152','153','166','194','202','233','263') THEN 'Upper East Side'
        WHEN pickup_location_id IN ('24','41','42','45','152','166') THEN 'Upper West Side'
        WHEN pickup_location_id IN ('4','224','211','229','232','238','262') THEN 'Uptown'
        ELSE 'Otra zona'
    END AS zona,
    COUNT(*) AS trip_count,
    AVG(fare_amount) AS avg_fare,
    AVG(trip_distance) AS avg_distance
FROM
    `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2015`
WHERE
    pickup_datetime BETWEEN '2015-01-01' AND '2015-12-31'
    AND fare_amount > 0
    AND fare_amount < 500
    AND trip_distance > 0
    AND pickup_location_id IS NOT NULL
GROUP BY
    fecha, hora, day_of_week, month, zona
HAVING zona != 'Otra zona'
ORDER BY
    fecha, hora, zona
"""

# Cache
cache_path = '../../../data/processed/nyc_taxi_zone_hour_2015.csv'
os.makedirs(os.path.dirname(cache_path), exist_ok=True)

if os.path.exists(cache_path):
    df = pd.read_csv(cache_path, parse_dates=['fecha'])
    print(f"Datos cargados desde cache: {len(df):,} registros")
else:
    df = bq.query_to_df(query)
    df['fecha'] = pd.to_datetime(df['fecha'])
    df.to_csv(cache_path, index=False)
    print(f"Datos descargados de BigQuery y guardados: {len(df):,} registros")

print(f"\nZonas: {df['zona'].unique()}")
print(f"Horas: {df['hora'].min()} a {df['hora'].max()}")
print(f"Rango de fechas: {df['fecha'].min().date()} a {df['fecha'].max().date()}")
print(f"\nViajes por zona:")
print(df.groupby('zona')['trip_count'].sum().sort_values(ascending=False).apply(lambda x: f'{x:,.0f}'))

## 3. Matriz de demanda: Zona x Hora del día

Este heatmap revela los **puntos calientes** de demanda: qué zonas tienen más
viajes a qué horas. Es la base para entender dónde posicionar los taxis.

In [ ]:
# Matriz de demanda promedio: zona x hora
demand_hour = df.groupby(['zona', 'hora'])['trip_count'].mean().reset_index()
demand_hour_pivot = demand_hour.pivot(index='zona', columns='hora', values='trip_count')

# Ordenar zonas por demanda total
zone_order = demand_hour_pivot.sum(axis=1).sort_values(ascending=False).index
demand_hour_pivot = demand_hour_pivot.loc[zone_order]

# Heatmap
fig, ax = plt.subplots(figsize=(18, 6))
sns.heatmap(demand_hour_pivot.astype(float), cmap='YlOrRd', annot=True, fmt='.0f',            linewidths=0.5, ax=ax, cbar_kws={'label': 'Viajes promedio por día'})
ax.set_title('Matriz de Demanda: Zona x Hora del Día (promedio diario)', 
             fontweight='bold', fontsize=14)
ax.set_xlabel('Hora del día')
ax.set_ylabel('Zona')
plt.tight_layout()
plt.show()

print("Observaciones clave:")
# Encontrar hora pico por zona
for zona in zone_order:
    peak_hour = demand_hour_pivot.loc[zona].idxmax()
    peak_val = demand_hour_pivot.loc[zona].max()
    print(f"  {zona}: hora pico = {int(peak_hour)}:00 ({peak_val:.0f} viajes promedio)")

## 4. Matriz de demanda: Zona x Día de la semana

Los patrones de demanda varían significativamente entre días laborables y fines de semana.
Esta información es crucial para planificar turnos de conductores.

In [ ]:
# Matriz de demanda promedio: zona x día de la semana
day_names = {1: 'Dom', 2: 'Lun', 3: 'Mar', 4: 'Mié', 5: 'Jue', 6: 'Vie', 7: 'Sáb'}
df['day_name'] = df['day_of_week'].map(day_names)

demand_day = df.groupby(['zona', 'day_of_week'])['trip_count'].mean().reset_index()
demand_day_pivot = demand_day.pivot(index='zona', columns='day_of_week', values='trip_count')
demand_day_pivot.columns = [day_names[d] for d in demand_day_pivot.columns]

# Reordenar: Lun a Dom
day_order = ['Lun', 'Mar', 'Mié', 'Jue', 'Vie', 'Sáb', 'Dom']
demand_day_pivot = demand_day_pivot[[d for d in day_order if d in demand_day_pivot.columns]]
demand_day_pivot = demand_day_pivot.loc[zone_order]

# Heatmap
fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(demand_day_pivot.astype(float), cmap='YlOrRd', annot=True, fmt='.0f',            linewidths=0.5, ax=ax, cbar_kws={'label': 'Viajes promedio por hora'})
ax.set_title('Matriz de Demanda: Zona x Día de la Semana (promedio por hora)',
             fontweight='bold', fontsize=14)
ax.set_xlabel('Día de la semana')
ax.set_ylabel('Zona')
plt.tight_layout()
plt.show()

# Ratio fin de semana / día laborable por zona
weekday_cols = [d for d in ['Lun', 'Mar', 'Mié', 'Jue', 'Vie'] if d in demand_day_pivot.columns]
weekend_cols = [d for d in ['Sáb', 'Dom'] if d in demand_day_pivot.columns]

print("\nRatio fin de semana / día laborable:")
for zona in zone_order:
    weekday_avg = demand_day_pivot.loc[zona, weekday_cols].mean()
    weekend_avg = demand_day_pivot.loc[zona, weekend_cols].mean()
    ratio = weekend_avg / weekday_avg
    print(f"  {zona}: {ratio:.2f}x {'(mas fines de semana)' if ratio > 1 else '(menos fines de semana)'}")

## 5. Feature engineering para predicción de demanda

Para predecir la demanda de la próxima hora en cada zona, creamos features que
capturan patrones temporales y rezagos:

- **Temporales:** hora, día de la semana, mes, es_fin_de_semana, es_feriado
- **Rezagos (lags):** demanda en t-1, t-2, t-24 (misma hora ayer)
- **Media móvil:** demanda promedio de los últimos 7 días para suavizar ruido

In [ ]:
# Preparar datos: necesitamos una serie por zona con frecuencia horaria
# Crear timestamp completo
df['timestamp'] = df['fecha'] + pd.to_timedelta(df['hora'], unit='h')

# Pivotar para tener una columna por zona
df_pivot = df.pivot_table(index='timestamp', columns='zona', 
                          values='trip_count', aggfunc='sum').fillna(0)

# Verificar
print(f"Shape de datos pivotados: {df_pivot.shape}")
print(f"Zonas: {list(df_pivot.columns)}")
print(f"Rango temporal: {df_pivot.index.min()} a {df_pivot.index.max()}")
df_pivot.head()

In [ ]:
# Definir feriados de EE.UU. 2015
us_holidays_2015 = pd.to_datetime([
    '2015-01-01', '2015-01-19', '2015-02-16', '2015-05-25',
    '2015-07-04', '2015-09-07', '2015-10-12', '2015-11-11',
    '2015-11-26', '2015-12-25'
])

def create_features(df_zone, zone_name):
    """Crea features temporales y rezagados para una zona."""
    feat = pd.DataFrame(index=df_zone.index)
    feat['demand'] = df_zone.values
    feat['zona'] = zone_name
    
    # Features temporales
    feat['hour'] = feat.index.hour
    feat['day_of_week'] = feat.index.dayofweek
    feat['month'] = feat.index.month
    feat['is_weekend'] = (feat['day_of_week'] >= 5).astype(int)
    feat['is_holiday'] = feat.index.normalize().isin(us_holidays_2015).astype(int)
    
    # Features cíclicos (para capturar la naturaleza circular de horas/días)
    feat['hour_sin'] = np.sin(2 * np.pi * feat['hour'] / 24)
    feat['hour_cos'] = np.cos(2 * np.pi * feat['hour'] / 24)
    feat['dow_sin'] = np.sin(2 * np.pi * feat['day_of_week'] / 7)
    feat['dow_cos'] = np.cos(2 * np.pi * feat['day_of_week'] / 7)
    
    # Rezagos (lags)
    feat['lag_1'] = feat['demand'].shift(1)     # hace 1 hora
    feat['lag_2'] = feat['demand'].shift(2)     # hace 2 horas
    feat['lag_24'] = feat['demand'].shift(24)   # misma hora ayer
    
    # Media móvil (7 días * 24 horas = 168 horas)
    feat['rolling_mean_7d'] = feat['demand'].rolling(window=168, min_periods=24).mean()
    
    # Diferencia respecto a la misma hora ayer
    feat['diff_24'] = feat['demand'] - feat['lag_24']
    
    return feat

# Crear features para todas las zonas
all_features = []
for zona in df_pivot.columns:
    feat = create_features(df_pivot[zona], zona)
    all_features.append(feat)

df_features = pd.concat(all_features)
df_features = df_features.dropna()  # Eliminar filas con NaN por los rezagos

print(f"Dataset con features: {len(df_features):,} registros")
print(f"Features creados: {len(df_features.columns)} columnas")
print(f"\nColumnas: {list(df_features.columns)}")
df_features.head(10)

## 6. Entrenamiento de Random Forest Regressor

Usamos **Random Forest** porque:
- Maneja bien relaciones no lineales
- No requiere normalización de features
- Proporciona importancia de features
- Es robusto ante outliers

Dividimos cronológicamente: enero-septiembre para entrenar, octubre-diciembre para evaluar.

In [ ]:
# Codificar zona como numérica
le = LabelEncoder()
df_features['zona_encoded'] = le.fit_transform(df_features['zona'])

# Features para el modelo
feature_cols = ['hour', 'day_of_week', 'month', 'is_weekend', 'is_holiday',
                'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
                'lag_1', 'lag_2', 'lag_24', 'rolling_mean_7d', 'diff_24',
                'zona_encoded']

X = df_features[feature_cols]
y = df_features['demand']

# Split cronológico
split_date = '2015-10-01'
train_mask = df_features.index < split_date
test_mask = df_features.index >= split_date

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

print(f"Entrenamiento: {X_train.shape[0]:,} registros (hasta {split_date})")
print(f"Prueba:        {X_test.shape[0]:,} registros (desde {split_date})")

# Entrenar Random Forest
print("\nEntrenando Random Forest Regressor...")
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)
print("Modelo entrenado.")

# Predicciones
y_pred_train = rf_model.predict(X_train)
y_pred_test = rf_model.predict(X_test)

# Métricas globales
mae_train = mean_absolute_error(y_train, y_pred_train)
mae_test = mean_absolute_error(y_test, y_pred_test)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
mape_test = np.mean(np.abs((y_test - y_pred_test) / y_test.replace(0, np.nan)).dropna()) * 100

print(f"\nMétricas globales:")
print(f"  MAE entrenamiento: {mae_train:,.1f} viajes")
print(f"  MAE prueba:        {mae_test:,.1f} viajes")
print(f"  RMSE prueba:       {rmse_test:,.1f} viajes")
print(f"  MAPE prueba:       {mape_test:.1f}%")

## 7. Importancia de features

¿Qué variables son más importantes para predecir la demanda en cada zona?
Esto nos dice qué factores controlan la demanda.

In [ ]:
# Importancia global de features
importances = pd.Series(rf_model.feature_importances_, index=feature_cols)
importances = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#FF5722' if imp > importances.mean() else '#2196F3' for imp in importances]
importances.plot(kind='barh', ax=ax, color=colors)
ax.set_title('Importancia de Features - Random Forest (Global)', fontweight='bold')
ax.set_xlabel('Importancia')
ax.axvline(importances.mean(), color='red', linestyle='--', alpha=0.5, label='Media')
ax.legend()
plt.tight_layout()
plt.show()

print("Interpretación:")
print(f"  Top 3 features: {', '.join(importances.tail(3).index.tolist())}")
print("  Los rezagos (demanda reciente) son los predictores más fuertes,")
print("  lo que confirma que la demanda tiene fuerte inercia temporal.")

In [ ]:
# Importancia por zona: entrenar un modelo por zona y comparar
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

zone_models = {}
for i, zona in enumerate(le.classes_):
    # Filtrar datos de esta zona
    mask_zona = df_features['zona'] == zona
    X_zona = df_features.loc[mask_zona, feature_cols].drop(columns=['zona_encoded'])
    y_zona = df_features.loc[mask_zona, 'demand']
    
    # Entrenar modelo específico
    rf_zona = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
    rf_zona.fit(X_zona[X_zona.index < split_date], y_zona[y_zona.index < split_date])
    zone_models[zona] = rf_zona
    
    # Graficar importancias
    if i < len(axes):
        imp = pd.Series(rf_zona.feature_importances_, 
                       index=[c for c in feature_cols if c != 'zona_encoded'])
        imp.sort_values().tail(8).plot(kind='barh', ax=axes[i], color='#FF5722', alpha=0.8)
        axes[i].set_title(f'{zona}', fontweight='bold', fontsize=11)
        axes[i].set_xlabel('Importancia')

# Ocultar ejes sobrantes
for j in range(len(le.classes_), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Top 8 Features por Zona', fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 8. Predicción de demanda de la próxima hora

Simulamos un escenario operativo: dado el estado actual, ¿cuánta demanda
habrá en cada zona en la próxima hora?

In [ ]:
# Usar las últimas observaciones del test set para simular predicción operativa
# Tomamos un día específico para ilustrar
sample_date = '2015-11-10'  # Un martes típico
sample_mask = (df_features.index.date == pd.Timestamp(sample_date).date())
sample_data = df_features[sample_mask].copy()

if len(sample_data) > 0:
    # Predicciones para cada hora del día
    sample_data['predicted'] = rf_model.predict(sample_data[feature_cols])
    
    # Visualizar real vs predicho por zona
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    
    for i, zona in enumerate(le.classes_):
        zona_data = sample_data[sample_data['zona'] == zona].sort_index()
        if len(zona_data) > 0 and i < len(axes):
            axes[i].plot(zona_data.index.hour if hasattr(zona_data.index, 'hour') else range(len(zona_data)),
                        zona_data['demand'].values, color='#2196F3', linewidth=2,
                        marker='o', markersize=4, label='Real')
            axes[i].plot(zona_data.index.hour if hasattr(zona_data.index, 'hour') else range(len(zona_data)),
                        zona_data['predicted'].values, color='#FF5722', linewidth=2,
                        marker='s', markersize=4, linestyle='--', label='Predicción')
            axes[i].set_title(f'{zona}', fontweight='bold')
            axes[i].set_xlabel('Hora')
            axes[i].set_ylabel('Viajes')
            axes[i].legend(fontsize=9)
    
    for j in range(len(le.classes_), len(axes)):
        axes[j].set_visible(False)
    
    plt.suptitle(f'Demanda Real vs Predicha - {sample_date} (Martes)', 
                 fontweight='bold', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print(f"No hay datos para la fecha {sample_date}. Ajuste la fecha de ejemplo.")

## 9. Recomendaciones de reposicionamiento de flota

Usando las predicciones de demanda, identificamos:
1. **Zonas con exceso de oferta:** donde la demanda predicha es baja respecto a la media
2. **Zonas con alta demanda predicha:** donde necesitamos más taxis
3. **Flujos recomendados:** mover taxis de zonas con exceso a zonas con déficit

In [ ]:
# Simular reposicionamiento para un momento específico
# Hora pico de la tarde: 18:00
target_hour = 18

# Demanda promedio por zona a las 18:00 (como proxy de oferta actual)
avg_demand_18 = df_features[
    (df_features['hour'] == target_hour) & 
    (df_features.index < split_date)
].groupby('zona')['demand'].mean()

# Predicción para el próximo día a las 18:00
pred_sample = df_features[
    (df_features['hour'] == target_hour) & 
    (df_features.index >= split_date)
].copy()

if len(pred_sample) > 0:
    pred_sample['predicted'] = rf_model.predict(pred_sample[feature_cols])
    avg_predicted = pred_sample.groupby('zona')['predicted'].mean()
    
    # Calcular déficit/exceso
    reposition = pd.DataFrame({
        'Demanda Histórica Promedio': avg_demand_18,
        'Demanda Predicha': avg_predicted,
        'Diferencia': avg_predicted - avg_demand_18,
        'Diferencia %': ((avg_predicted - avg_demand_18) / avg_demand_18 * 100)
    }).round(1)
    
    reposition = reposition.sort_values('Diferencia', ascending=False)
    
    print(f"ANÁLISIS DE REPOSICIONAMIENTO - Hora: {target_hour}:00")
    print("=" * 70)
    print(reposition.to_string())
    print(f"\nZonas con exceso (mover taxis DESDE aquí):")
    excess = reposition[reposition['Diferencia'] < 0]
    for zona in excess.index:
        print(f"  {zona}: exceso de ~{abs(excess.loc[zona, 'Diferencia']):.0f} viajes/hora")
    
    print(f"\nZonas con déficit (mover taxis HACIA aquí):")
    deficit = reposition[reposition['Diferencia'] > 0]
    for zona in deficit.index:
        print(f"  {zona}: déficit de ~{deficit.loc[zona, 'Diferencia']:.0f} viajes/hora")

In [ ]:
# Tabla de recomendaciones de flujo
print("\nRECOMENDACIONES DE REPOSICIONAMIENTO DE FLOTA")
print("=" * 70)

if len(excess) > 0 and len(deficit) > 0:
    recommendations = []
    
    # Crear pares de origen-destino
    excess_sorted = excess.sort_values('Diferencia')  # más negativo primero
    deficit_sorted = deficit.sort_values('Diferencia', ascending=False)  # más positivo primero
    
    remaining_excess = abs(excess_sorted['Diferencia']).to_dict()
    remaining_deficit = deficit_sorted['Diferencia'].to_dict()
    
    for from_zone in remaining_excess:
        for to_zone in remaining_deficit:
            if remaining_excess[from_zone] > 0 and remaining_deficit[to_zone] > 0:
                taxis_to_move = min(remaining_excess[from_zone], remaining_deficit[to_zone])
                if taxis_to_move > 5:  # Solo recomendar movimientos significativos
                    recommendations.append({
                        'Desde': from_zone,
                        'Hacia': to_zone,
                        'Taxis a mover': int(taxis_to_move),
                        'Prioridad': 'Alta' if taxis_to_move > 50 else 'Media'
                    })
                    remaining_excess[from_zone] -= taxis_to_move
                    remaining_deficit[to_zone] -= taxis_to_move
    
    if recommendations:
        rec_df = pd.DataFrame(recommendations)
        print(rec_df.to_string(index=False))
    else:
        print("No se requieren movimientos significativos en este momento.")
else:
    print("El balance de oferta-demanda es adecuado en este momento.")

# Visualización de flujos
fig, ax = plt.subplots(figsize=(14, 6))
zones = reposition.index
colors = ['#F44336' if d < 0 else '#4CAF50' for d in reposition['Diferencia']]
bars = ax.bar(zones, reposition['Diferencia'], color=colors, edgecolor='white', linewidth=1.5)
ax.axhline(0, color='black', linewidth=1)
ax.set_title(f'Déficit/Exceso de Demanda por Zona - {target_hour}:00h', fontweight='bold', fontsize=14)
ax.set_ylabel('Diferencia (predicha - histórica)')
ax.set_xlabel('Zona')

for bar, val in zip(bars, reposition['Diferencia']):
    y_pos = bar.get_height() + (2 if val >= 0 else -8)
    ax.text(bar.get_x() + bar.get_width()/2, y_pos, f'{val:+.0f}',
            ha='center', fontweight='bold', fontsize=11)

# Leyenda
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#4CAF50', label='Necesita más taxis'),
                   Patch(facecolor='#F44336', label='Exceso de taxis')]
ax.legend(handles=legend_elements, loc='upper right')

plt.tight_layout()
plt.show()

## 10. Evaluación por zona: ¿Qué tan precisas son las predicciones?

Es importante saber en qué zonas el modelo funciona mejor,
ya que las recomendaciones solo son confiables si las predicciones son precisas.

In [ ]:
# Métricas por zona
test_data = df_features[test_mask].copy()
test_data['predicted'] = rf_model.predict(X_test)

metrics_by_zone = []
for zona in le.classes_:
    zona_mask = test_data['zona'] == zona
    actual = test_data.loc[zona_mask, 'demand']
    predicted = test_data.loc[zona_mask, 'predicted']
    
    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    # MAPE evitando división por cero
    non_zero = actual > 0
    mape = np.mean(np.abs((actual[non_zero] - predicted[non_zero]) / actual[non_zero])) * 100
    
    metrics_by_zone.append({
        'Zona': zona,
        'MAE': mae,
        'RMSE': rmse,
        'MAPE (%)': mape,
        'Demanda Media': actual.mean()
    })

metrics_zone_df = pd.DataFrame(metrics_by_zone).sort_values('MAPE (%)')

print("MÉTRICAS DE PREDICCIÓN POR ZONA")
print("=" * 65)
print(metrics_zone_df.to_string(index=False))

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# MAE por zona
metrics_zone_df_sorted = metrics_zone_df.sort_values('MAE')
axes[0].barh(metrics_zone_df_sorted['Zona'], metrics_zone_df_sorted['MAE'], color='#2196F3')
axes[0].set_title('MAE por Zona', fontweight='bold')
axes[0].set_xlabel('Error Absoluto Medio (viajes)')

# MAPE por zona
metrics_zone_df_sorted = metrics_zone_df.sort_values('MAPE (%)')
colors_mape = ['#4CAF50' if m < 20 else '#FF9800' if m < 30 else '#F44336' 
               for m in metrics_zone_df_sorted['MAPE (%)']]
axes[1].barh(metrics_zone_df_sorted['Zona'], metrics_zone_df_sorted['MAPE (%)'], color=colors_mape)
axes[1].set_title('MAPE por Zona', fontweight='bold')
axes[1].set_xlabel('Error Porcentual Absoluto Medio (%)')
axes[1].axvline(20, color='green', linestyle='--', alpha=0.5, label='Umbral aceptable (20%)')
axes[1].legend()

plt.suptitle('Precisión del Modelo de Predicción de Demanda por Zona',
             fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 11. Estimación de valor de negocio

¿Cuánto ingreso adicional podría generar un sistema de reposicionamiento óptimo?

Estimamos de forma conservadora:
- Cada viaje no servido por falta de taxi disponible representa ingreso perdido
- Si mejoramos la distribución de la flota un 5-10%, reducimos viajes perdidos

In [ ]:
# Estimación de valor de negocio
avg_fare_per_trip = df['avg_fare'].mean()
total_daily_trips = df.groupby('fecha')['trip_count'].sum().mean()

print("ESTIMACIÓN DE VALOR DE NEGOCIO")
print("=" * 60)
print(f"\nSupuestos:")
print(f"  Tarifa promedio por viaje: ${avg_fare_per_trip:.2f}")
print(f"  Viajes diarios promedio: {total_daily_trips:,.0f}")

# Escenarios de mejora
scenarios = [0.02, 0.05, 0.10]  # 2%, 5%, 10% mejora en captura de demanda

print(f"\n{'Escenario':<25} {'Viajes adicionales/día':>22} {'Ingreso adicional/día':>22} {'Ingreso adicional/año':>22}")
print("-" * 95)

for scenario in scenarios:
    additional_trips = total_daily_trips * scenario
    additional_revenue_day = additional_trips * avg_fare_per_trip
    additional_revenue_year = additional_revenue_day * 365
    
    print(f"  Mejora del {scenario*100:.0f}%{'':<16} {additional_trips:>18,.0f}     ${additional_revenue_day:>18,.0f}     ${additional_revenue_year:>18,.0f}")

print(f"\nIncluso una mejora del 2% en la captura de demanda puede generar")
print(f"millones de dólares adicionales al año.")
print(f"\nNota: Esta estimación es conservadora. No incluye:")
print(f"  - Reducción de tiempos de espera (mejora satisfacción del cliente)")
print(f"  - Mayor eficiencia de combustible (menos taxis circulando vacíos)")
print(f"  - Efecto de retención de conductores (más ingresos = menos rotación)")

## 12. Guardar modelo y resultados

In [ ]:
# Guardar modelo y artefactos
model_dir = '../../../models/nyc_taxi/'
os.makedirs(model_dir, exist_ok=True)

# Modelo Random Forest
with open(os.path.join(model_dir, 'demand_rf_model.pkl'), 'wb') as f:
    pickle.dump(rf_model, f)

# Label encoder para zonas
with open(os.path.join(model_dir, 'zone_encoder.pkl'), 'wb') as f:
    pickle.dump(le, f)

# Métricas por zona
metrics_zone_df.to_csv(os.path.join(model_dir, 'demand_metrics_by_zone.csv'), index=False)

# Feature importances
importances.to_csv(os.path.join(model_dir, 'demand_feature_importances.csv'))

print(f"Artefactos guardados en: {os.path.abspath(model_dir)}")
print(f"  - demand_rf_model.pkl: Modelo Random Forest")
print(f"  - zone_encoder.pkl: Codificador de zonas")
print(f"  - demand_metrics_by_zone.csv: Métricas por zona")
print(f"  - demand_feature_importances.csv: Importancia de features")

## 13. Resumen y consideraciones de implementación

### Resultados principales

1. **Midtown** es consistentemente la zona de mayor demanda, especialmente en horas pico (8-9am, 6-7pm)
2. **Los rezagos temporales** (demanda reciente) son los predictores más importantes, lo que confirma que la demanda tiene fuerte inercia
3. **El modelo Random Forest** logra predicciones razonables, con MAPE variable según la zona
4. **El reposicionamiento proactivo** tiene potencial de generar millones en ingreso adicional

### Consideraciones prácticas para implementación

| Aspecto | Consideración |
|---------|---------------|
| Latencia | El modelo debe predecir en < 1 segundo para ser útil en tiempo real |
| Frecuencia | Actualizar predicciones cada 15-30 minutos |
| Datos en tiempo real | Necesita feed de posición/disponibilidad de taxis |
| Incentivos | Los conductores no se mueven gratis: ofrecer bonus por reposicionamiento |
| Granularidad | Zonas TLC (1-263) ofrecen buena granularidad para la logística |
| Eventos especiales | Integrar calendarios de eventos (conciertos, deportes) como features |
| Clima | Incorporar pronóstico del tiempo como variable predictora |

### Limitaciones del análisis

- Las zonas geográficas se basan en los TLC Zone IDs (pickup_location_id)
- No tenemos datos de oferta real (posición de taxis disponibles)
- El modelo asume patrones estables, pero la entrada de Uber/Lyft cambió el mercado
- Los datos de 2015 pueden no reflejar la dinámica actual

### Próximos pasos del proyecto

- Fase 6: Dashboard interactivo para visualizar predicciones y recomendaciones en tiempo real